In [ ]:
import ee
import json
import os
import datetime as dt
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, TileLayer, LayersControl, GeoJSON, WidgetControl
from IPython.display import display
import requests
import base64
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import numpy as np
from ipyleaflet import Popup
import ipywidgets as widgets

# --------------------
# EE init
# --------------------
key_dict = json.loads(os.environ["GEE_SERVICE_ACCOUNT_KEY"])
with open("service_account.json", "w") as f:
    json.dump(key_dict, f)

credentials = ee.ServiceAccountCredentials(
    email=key_dict["client_email"],
    key_file="service_account.json"
)
ee.Initialize(credentials=credentials, project="ee-joshtao2000-482701")

In [ ]:
import requests
import os
import ipywidgets as widgets
from IPython.display import display

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json"
}

province_options = [
    "-- Select --",
    "Alberta",
    "British Columbia",
    "Manitoba",
    "New Brunswick",
    "Newfoundland and Labrador",
    "Nova Scotia",
    "Northwest Territories",
    "Nunavut",
    "Ontario",
    "Prince Edward Island",
    "Quebec",
    "Saskatchewan",
    "Yukon",
    "Outside Canada"
]

purpose_options = [
    "-- Select --",
    "Research",
    "Education",
    "Environmental Monitoring",
    "Personal Interest",
    "Government/Policy",
    "Other"
]

# ====================
# UI
# ====================
auth_title = widgets.HTML(
    value="<h2 style='color:#2b83ba;font-family:Arial;'>🌊 Northern Lakes Monitor</h2>"
)

tab = widgets.Tab()

# --- Register tab ---
reg_name = widgets.Text(description="Name:", placeholder="Your name",
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
reg_email = widgets.Text(description="Email:", placeholder="your@email.com",
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
reg_province = widgets.Dropdown(description="Province/Territory:",
    options=province_options,
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
reg_city = widgets.Text(description="City (optional):", placeholder="Your city",
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
reg_purpose = widgets.Dropdown(description="Purpose:",
    options=purpose_options,
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
reg_btn = widgets.Button(description="Register & Enter", button_style="primary",
    layout=widgets.Layout(width="150px"))
reg_status = widgets.HTML()

register_box = widgets.VBox([
    widgets.HTML("<p style='font-family:Arial;color:#555;'>New user? Please register to access the tool.</p>"),
    reg_name, reg_email, reg_province, reg_city, reg_purpose,
    reg_btn, reg_status
], layout=widgets.Layout(padding="20px"))

# --- Login tab ---
login_email = widgets.Text(description="Email:", placeholder="your@email.com",
    style={"description_width": "100px"}, layout=widgets.Layout(width="350px"))
login_btn = widgets.Button(description="Enter", button_style="primary",
    layout=widgets.Layout(width="150px"))
login_status = widgets.HTML()

login_box = widgets.VBox([
    widgets.HTML("<p style='font-family:Arial;color:#555;'>Already registered? Enter your email to continue.</p>"),
    login_email, login_btn, login_status
], layout=widgets.Layout(padding="20px"))

tab.children = [login_box,register_box]
tab.set_title(1, "Register")
tab.set_title(0, "Login")

intro_html = widgets.HTML(value="""
<div style='font-family:Arial;font-size:13px;line-height:1.8;color:#333;
    padding:20px;max-width:500px;'>

    <img src='https://i.imgur.com/Bsv4Pn3.png' 
         style='width:100%;border-radius:8px;margin-bottom:8px;
         border:1px solid #ddd;'>
    <p style='font-size:11px;color:#888;margin-bottom:16px;font-style:italic;'>
    Screenshot of the Watching Water Quality tool showing Chl-a levels near Yellowknife, NWT.</p>

    <h3 style='color:#2b83ba;margin-bottom:8px;'>Why Water Quality Matters</h3>
    <p>Lakes and rivers are essential to life for drinking, fishing, swimming, 
    and supporting ecosystems. When water quality declines, it affects human health, 
    wildlife, and the communities that depend on these water bodies.</p>

    <h3 style='color:#2b83ba;margin-top:16px;margin-bottom:8px;'>What Is Chlorophyll-a?</h3>
    <p>Chlorophyll-a (Chl-a) is the green pigment produced by algae and 
    phytoplankton in lakes. When algae grow excessively, they form harmful algal 
    blooms (HABs) that can poison drinking water, kill fish, and make lakes unsafe 
    for swimming. Measuring Chl-a concentration, in micrograms per litre (µg/L), 
    tells us how much algae is present and how at-risk a lake may be.</p>

    <h3 style='color:#2b83ba;margin-top:16px;margin-bottom:8px;'>About This Tool</h3>
    <p>This tool uses Sentinel-2 satellite imagery and a machine-learning model 
    to estimate Chl-a levels across lakes in Canada's North. It allows anyone, including 
    researchers, resource managers, and concerned citizens, to monitor lake 
    health without needing to collect water samples.</p>

    <img src='https://i.imgur.com/iZYEI6E.png' 
         style='width:100%;border-radius:8px;margin-top:16px;margin-bottom:8px;
         border:1px solid #ddd;'>
    <p style='font-size:11px;color:#666;margin-bottom:16px;font-style:italic;'>
    Satellite-derived chlorophyll-a (Chl-a) maps for Yellowknife Bay in July (2020–2025). 
    Some warmer years, such as 2021 and 2024, show higher algae levels in parts of the bay. 
    These maps are designed to highlight patterns and changes over time and are not a 
    replacement for water quality testing.</p>

    <p style='margin-top:8px;padding:10px;background:#fff8e1;
    border-left:4px solid #f97b06;border-radius:4px;font-size:12px;color:#555;'>
    ⚠️ This tool was trained on lakes in the Northwest Territories and Yukon. 
    Results in other regions should be treated as relative indicators only, 
    not a substitute for laboratory water quality testing.
    </p>

</div>
""")

auth_box = widgets.VBox([
    auth_title,
    widgets.HBox([
        intro_html,
        widgets.HTML("<div style='width:1px;background:#ddd;margin:0 20px;'></div>"),
        tab
    ], layout=widgets.Layout(align_items="flex-start"))
])


#auth_box = widgets.VBox([auth_title, tab])

# Main app placeholder
main_app = widgets.Output()

# ====================
# Logic
# ====================
def check_user(email):
    r = requests.get(
        f"{SUPABASE_URL}/rest/v1/users?email=eq.{email}&select=id,login_count",
        headers=HEADERS
    )
    data = r.json()
    if isinstance(data, list) and len(data) > 0:
        return data
    return None

def register_user(name, email, province, city, purpose):
    r = requests.post(
        f"{SUPABASE_URL}/rest/v1/users",
        headers=HEADERS,
        json={
            "name": name,
            "email": email,
            "province_territory": province,
            "city": city,
            "purpose": purpose
        }
    )
    return r.status_code

def update_login(email, current_count):
    requests.patch(
        f"{SUPABASE_URL}/rest/v1/users?email=eq.{email}",
        headers=HEADERS,
        json={
            "last_login": "now()",
            "login_count": current_count + 1
        }
    )

def launch_app():
    auth_box.layout.display = "none"
    with main_app:
        display(app)  # 你现有的app widget

def on_register(b):
    name = reg_name.value.strip()
    email = reg_email.value.strip()
    province = reg_province.value
    purpose = reg_purpose.value

    if not name:
        reg_status.value = "<span style='color:red'>⚠️ Please enter your name.</span>"
        return
    if not email or "@" not in email:
        reg_status.value = "<span style='color:red'>⚠️ Please enter a valid email.</span>"
        return
    if province == "-- Select --":
        reg_status.value = "<span style='color:red'>⚠️ Please select your province/territory.</span>"
        return
    if purpose == "-- Select --":
        reg_status.value = "<span style='color:red'>⚠️ Please select your purpose.</span>"
        return

    existing = check_user(email)
    if existing is not None:
        reg_status.value = "<span style='color:orange'>⚠️ Email already registered. Please use the Returning User tab.</span>"
        return

    status_code = register_user(name, email, province, reg_city.value.strip(), purpose)
    if status_code == 201:
        reg_status.value = "<span style='color:green'>✅ Registered! Loading tool...</span>"
        launch_app()
    else:
        reg_status.value = f"<span style='color:red'>❌ Registration failed (status {status_code}). Please try again.</span>"



def on_login(b):
    email = login_email.value.strip()
    if not email or "@" not in email:
        login_status.value = "<span style='color:red'>⚠️ Please enter a valid email.</span>"
        return

    existing = check_user(email)
    if not existing:
        login_status.value = "<span style='color:red'>⚠️ Email not found. Please register first.</span>"
        return

    update_login(email, existing[0]["login_count"])
    login_status.value = "<span style='color:green'>✅ Welcome back! Loading tool...</span>"
    launch_app()

reg_btn.on_click(on_register)
login_btn.on_click(on_login)

display(auth_box, main_app)

In [ ]:
# ====================
# State
# ====================
state = {"aoi_geom": None, "rgb_layer": None, "chla_layer": None, "aoi_layer": None}

# ====================
# Map
# ====================
m = Map(center=(62.45, -114.37), zoom=8, scroll_wheel_zoom=True)
m.layout.height = "calc(100vh - 60px)"
m.layout.width = "100%"

draw = DrawControl(
    circlemarker={}, marker={}, polyline={}, circle={},
    polygon={"shapeOptions": {"color": "#2b83ba"}},
    rectangle={"shapeOptions": {"color": "#2b83ba"}},
)
m.add_control(draw)
m.add_control(LayersControl(position="topright"))

# ====================
# Widgets
# ====================
date_picker = widgets.DatePicker(
    description="Date:",
    value=dt.date(2024, 6, 15),
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px")
)

#date_picker = widgets.Text(
#    description="Date:",
#    value="2024-06-15",
#    placeholder="YYYY-MM-DD",
#    style={"description_width": "80px"},
#    layout=widgets.Layout(width="280px")
#)

window_days = widgets.IntSlider(
    description="± Days:",
    min=0, max=30, value=10,
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px")
)
cloud_pct = widgets.IntSlider(
    description="Max Cloud %:",
    min=0, max=100, value=30,
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px")
)
#show_rgb = widgets.Checkbox(description="Show Satellite Image", value=True)
#show_chla = widgets.Checkbox(description="Show Chl-a map", value=True)
show_rgb = widgets.Checkbox(description="Show Satellite Image", value=True,
    layout=widgets.Layout(display="none"))
show_chla = widgets.Checkbox(description="Show Chl-a map", value=True,
    layout=widgets.Layout(display="none"))
run_btn = widgets.Button(description="Run", button_style="primary", layout=widgets.Layout(width="120px"))
reset_btn = widgets.Button(description="Reset", button_style="primary",layout=widgets.Layout(width="120px"))
export_btn = widgets.Button(description="Export Map", button_style="primary", layout=widgets.Layout(width="120px"))
status = widgets.HTML(value="<b>Draw an AOI (rectangle/polygon) on the map, then click Run.</b>")
click_output = widgets.HTML(value="<b>Click on water to get Chl-a value</b>")


#about_btn = widgets.Button(
#    description="About",
#    button_style="primary",
#    layout=widgets.Layout(width="120px")
#)

#about_content = widgets.VBox([
#    widgets.HTML(value="""
#    <div style="background:white;border:2px solid #0a1628;border-radius:10px;padding:16px;font-family:Arial;font-size:13px;margin-top:8px;line-height:1.7;">
#    <b style="font-size:15px;color:#0a1628;">About This Tool</b>
#    <p style="margin:10px 0 0;">Water quality describes the physical, chemical, and biological characteristics of water that determine its suitability for drinking, swimming, fishing, and supporting aquatic life. Good water quality means water is clean, clear, and safe. Poor water quality can harm human health, damage ecosystems, and affect the livelihoods of communities that depend on lakes and rivers.</p>
#    <p style="margin:10px 0 0;">Chlorophyll-a (Chl-a) is the green pigment found in all photosynthesising organisms, including phytoplankton and algae in lakes. Scientists measure Chl-a concentration in micrograms per litre (µg/L) as an indicator of how much algae is present. Higher values signal a shift toward poorer water quality or an approaching algal bloom.</p>
#    <p style="margin:10px 0 0;">We developed this tool to estimate chlorophyll-a levels by using Sentinel-2 satellite imagery and a Random Forest machine-learning model.</p>
#    <p style="margin:10px 0 0;color:#dc2626;font-size:12px;">⚠️ Model trained on NWT & Yukon lakes. For other regions, values are relative indicators only — not a replacement for laboratory water quality testing.</p>
#    </div>
#    """),
#    widgets.Button(description="Close", button_style="warning", layout=widgets.Layout(width="80px"))
#], layout=widgets.Layout(display="none"))

#about_content.children[1]._click_handlers.callbacks.clear()
#about_content.children[1].on_click(lambda b: setattr(about_content.layout, 'display', 'none'))



# About button
about_btn = widgets.Button(
    description="ℹ️ About",
    style=widgets.ButtonStyle(button_color="#2b83ba"),
    layout=widgets.Layout(width="120px", height="32px")
)

# About popup - 覆盖在地图上方
about_popup = widgets.HTML(value="", layout=widgets.Layout(
    width="320px",
    max_width="320px",
    overflow="hidden",
    display="none"
))


about_popup_content = """
<div style='background:white;border:2px solid #2b83ba;border-radius:8px;
    padding:12px;font-family:Arial;font-size:11px;box-shadow:2px 2px 6px rgba(0,0,0,0.2);
    width:295px;max-width:295px;overflow:hidden;box-sizing:border-box;'>
    <b style='color:#2b83ba;font-size:13px;'>About This Tool</b>
    <p style="margin:10px 0 0;">Water quality describes the physical, chemical, and biological characteristics of water that determine its suitability for drinking, swimming, fishing, and supporting aquatic life. Good water quality means water is clean, clear, and safe. Poor water quality can harm human health, damage ecosystems, and affect the livelihoods of communities that depend on lakes and rivers.</p>
    <p style="margin:10px 0 0;">Chlorophyll-a (Chl-a) is the green pigment found in all photosynthesising organisms, including phytoplankton and algae in lakes. Scientists measure Chl-a concentration in micrograms per litre (µg/L) as an indicator of how much algae is present. Higher values signal a shift toward poorer water quality or an approaching algal bloom.</p>
    <p style="margin:10px 0 0;">We developed this tool to estimate chlorophyll-a levels by using Sentinel-2 satellite imagery and a Random Forest machine-learning model.</p>
    <p style="margin:10px 0 0;color:#dc2626;font-size:12px;">⚠️ Model trained on NWT & Yukon lakes. For other regions, values are relative indicators only — not a replacement for laboratory water quality testing.</p>
</div>
"""




def on_about(b):
    if about_popup.layout.display == "none":
        about_popup.value = about_popup_content
        about_popup.layout.display = "block"
    else:
        about_popup.layout.display = "none"
        about_popup.value = ""

about_btn._click_handlers.callbacks.clear()
about_btn.on_click(on_about)



#def on_about(b):
#    if about_content.layout.display == "none":
#        about_content.layout.display = "block"
#    else:
#        about_content.layout.display = "none"

#about_btn._click_handlers.callbacks.clear()
#about_btn.on_click(on_about)

# Help button
help_visible = [False]
help_btn = widgets.Button(
    description="How to Use",
    style=widgets.ButtonStyle(button_color="#2b83ba"),
#    button_style="primary",
    layout=widgets.Layout(width="120px",height="32px")
)

help_output = widgets.HTML(value="", layout=widgets.Layout(display="none"))

def on_help(b):
    if help_output.layout.display == "none":
        help_output.layout.display = "block"
        help_output.value = """
        <div style="background:white;border:2px solid #0d9488;border-radius:10px;padding:16px;font-family:Arial;font-size:13px;margin-top:8px;">
        <b style="font-size:15px;color:#0a1628;">How to Use — click button again to close</b>
        <ol style="margin:10px 0 0 16px;line-height:2.2;">
          <li><b>Draw AOI</b> — Use the rectangle or polygon tool on the map</li>
          <li><b>Set date</b> — Enter date in YYYY-MM-DD format</li>
          <li><b>Adjust ± Days</b> — Search window around your date (default 10)</li>
          <li><b>Max Cloud %</b> — Increase if no scenes found (default 30)</li>
          <li><b>Click Run</b> — Wait 30–90 seconds for results</li>
          <li><b>Click water</b> — See exact Chl-a value in µg/L</li>
          <li><b>Export PNG</b> — Download map with legend</li>
          <li><b>Reset</b> — Clear and start a new analysis</li>
        </ol>
        <p style="margin:10px 0 0;color:#dc2626;font-size:12px;">⚠️ Model trained on NWT & Yukon lakes. For other regions, values are relative indicators only.</p>
        </div>
        """
    else:
        help_output.layout.display = "none"
        help_output.value = ""

help_btn._click_handlers.callbacks.clear()
help_btn.on_click(on_help)

# ====================
# AOI draw handler
# ====================
def on_draw(target, action, geo_json):
    if action in ("created", "edited"):
        geom = geo_json.get("geometry")
        if geom:
            state["aoi_geom"] = geom
            status.value = "<b style='color:green'>AOI set ✅ Click Run to process.</b>"
    elif action == "deleted":
        state["aoi_geom"] = None
        status.value = "<b>AOI deleted. Draw a new AOI.</b>"

draw.on_draw(on_draw)

# ====================
# EE Helpers
# ====================
def build_s2_collection(aoi, d, win, cloud):
    d0 = ee.Date(str(d)).advance(-win, "day")
    d1 = ee.Date(str(d)).advance(win, "day")
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(d0, d1)
            .filterBounds(aoi)
            .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", cloud)))

def mask_clouds(img):
    s2_cloudless = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    cloud_prob = ee.Image(
        s2_cloudless
        .filter(ee.Filter.eq('system:index', img.get('system:index')))
        .first()
    ).select('probability')
    scl = img.select('SCL')
    scl_mask = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                .And(scl.neq(10)).And(scl.neq(11)))
    qa60 = img.select('QA60')
    qa_mask = (qa60.bitwiseAnd(1 << 10).eq(0)
               .And(qa60.bitwiseAnd(1 << 11).eq(0)))
    
    b2 = img.select('B2').multiply(0.0001)
    b3 = img.select('B3').multiply(0.0001)
    cirrus_mask = b2.lt(0.25).And(b3.lt(0.25))
    cirrus_class = img.select('MSK_CLASSI_CIRRUS').eq(0)
    is_cloud = cloud_prob.gt(15)
    cloud_shadow_azimuth = ee.Number(90).subtract(
        ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))
    is_cloud_dilated = (is_cloud
        .directionalDistanceTransform(cloud_shadow_azimuth, 10)
        .reproject(crs=img.select('B2').projection(), scale=100)
        .select('distance')
        .mask()
        .rename('clouds'))
    return img.updateMask(
        scl_mask.And(qa_mask).And(cirrus_mask)
        .And(cirrus_class).And(is_cloud_dilated.Not())
    )

def compute_indices(img):
    b2 = img.select("B2").multiply(0.0001)
    b3 = img.select("B3").multiply(0.0001)
    b4 = img.select("B4").multiply(0.0001)
    b5 = img.select("B5").multiply(0.0001)
    b8 = img.select("B8").multiply(0.0001)
    b11 = img.select("B11").multiply(0.0001)
    ndvi = b8.subtract(b4).divide(b8.add(b4)).rename("NDVI")
    ndwi = b8.subtract(b3).divide(b8.add(b3)).rename("NDWI")
    mndwi = b3.subtract(b11).divide(b11.add(b3)).rename("MNDVI")
    ndci = b5.subtract(b4).divide(b5.add(b4)).rename("NDCI")
    ndti = b4.subtract(b3).divide(b4.add(b3)).rename("NDTI")
    tsm = b4.divide(b3).rename("TSM")
    cdom23 = b2.divide(b3).rename("CDOM_B2B3")
    cdom34 = b3.divide(b4).rename("CDOM_B3B4")
    cdom35 = b3.divide(b5).rename("CDOM_B3B5")
    return img.addBands([ndci, tsm, cdom23, cdom34, cdom35,ndti, ndwi, ndvi,mndwi], overwrite=True)

# ====================
# USER SETTINGS
# ====================
RF_MODEL_ASSET_ID = "users/YOUR_USERNAME/YOUR_RF_CHLA_MODEL"
PREDICTOR_BANDS = ["NDCI", "TSM", "CDOM", "NDWI","MNDVI"]

def load_rf_model():
    return ee.Classifier.load(
        'projects/ee-joshtao2000-482701/assets/RF_chla_model'
    ).setOutputMode('REGRESSION')


def rf_predict_chla(img_with_features):
#    PREDICTORS = ["NDCI", "NDTI", "NDWI", "CDOM_B3B4"]
    PREDICTORS = ["NDCI", "NDTI", "CDOM_B3B4","MNDVI"]
    TARGET = "log_chl_a"
    
    train_fc = ee.FeatureCollection("projects/ee-joshtao2000-482701/assets/s2_chla_l2a_slave_cal")\
    .filter(ee.Filter.notNull(PREDICTORS + [TARGET]))\
    .merge(
        ee.FeatureCollection("projects/ee-joshtao2000-482701/assets/s2_chla_l2a_brdf_val")\
        .filter(ee.Filter.notNull(PREDICTORS + [TARGET]))
    )


#    train_fc = ee.FeatureCollection("projects/ee-joshtao2000-482701/assets/s2_chla_l2a_slave_cal")\
#        .filter(ee.Filter.notNull(PREDICTORS + [TARGET]))
    
    rf = ee.Classifier.smileRandomForest(
        numberOfTrees=100,
        minLeafPopulation=2,
        bagFraction=0.8,
        seed=50
    ).setOutputMode("REGRESSION")
    
    model = rf.train(
        features=train_fc,
        classProperty=TARGET,
        inputProperties=PREDICTORS
    )

#    model=load_rf_model()
    
    log_chla = img_with_features.select(PREDICTORS).classify(model)
    chla = ee.Image(10).pow(log_chla).rename("chla")
    
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    permanent_water = jrc.select("occurrence").gte(30)
    water_mask_clean = permanent_water.focal_min(
        radius=1, kernelType='square', units='pixels'
    )
    ndwi = img_with_features.select("NDWI")
    water_mask_ndwi = ndwi.gt(0)
    
    # 两个掩膜取交集
    water_mask_combined = water_mask_clean.And(water_mask_ndwi)
    
    mndwi = img_with_features.select("MNDVI")  # 你的MNDWI命名为MNDVI
    water_mask_mndwi = mndwi.gt(0.1)  # 阈值0.1比0更严格
    water_mask_combined = water_mask_clean.And(water_mask_mndwi)
    chla = chla.updateMask(water_mask_combined)
#    chla = chla.updateMask(water_mask_clean)
    chla = chla.clamp(0, 30)
    return chla



def lr_predict_chla(img_with_features):
    a = float(os.environ["CHLA_A"])
    b = float(os.environ["CHLA_B"])

    ndci = img_with_features.select("NDCI")
    log_chla = ndci.multiply(a).add(b)
    chla = ee.Image(10).pow(log_chla).rename("chla")
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    permanent_water = jrc.select("occurrence").gte(40)
    water_mask_clean = permanent_water.focal_min(radius=1, kernelType='square', units='pixels')
    chla = chla.updateMask(water_mask_clean)
    chla = chla.clamp(0, 30)
    return chla


def mr_predict_chla(img_with_features):
    """
    Applies multiple linear regression model to produce Chl-a band.
    Assumes model expects PREDICTOR_BANDS present in the image.
    """

    a_ndci = float(os.environ["MR_COEF_NDCI"])
    a_ndti = float(os.environ["MR_COEF_NDTI"])
    a_cdom = float(os.environ["MR_COEF_CDOM"])
    a_ndwi = float(os.environ["MR_COEF_NDWI"])
    intercept = float(os.environ["MR_INTERCEPT"])
    
    ndci = img_with_features.select("NDCI")
    ndti = img_with_features.select("NDTI")
    cdom = img_with_features.select("CDOM_B3B4")
    ndwi = img_with_features.select("NDWI")
    
    log_chla = (ndci.multiply(a_ndci)
                .add(ndti.multiply(a_ndti))
                .add(cdom.multiply(a_cdom))
                .add(ndwi.multiply(a_ndwi))
                .add(intercept))
    
    chla = ee.Image(10).pow(log_chla).rename("chla")
    
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
    permanent_water = jrc.select("occurrence").gte(60)
    water_mask_clean = permanent_water.focal_min(
        radius=1, kernelType='square', units='pixels'
    )

    ndwi = img_with_features.select("NDWI")
    water_mask_ndwi = ndwi.gt(0)
    
    # 两个掩膜取交集
    water_mask_combined = water_mask_clean.And(water_mask_ndwi)
    
    mndwi = img_with_features.select("MNDVI")  # 你的MNDWI命名为MNDVI
    water_mask_mndwi = mndwi.gt(0.3)  # 阈值0.1比0更严格
    water_mask_combined = water_mask_clean.And(water_mask_mndwi)
    chla = chla.updateMask(water_mask_combined)
#    chla = chla.updateMask(water_mask_clean)
    chla = chla.clamp(0, 30)
    return chla


def get_ee_tile_url(img, vis):
    mapid = ee.Image(img).getMapId(vis)
    return mapid["tile_fetcher"].url_format

def clear_layer(key):
    layer = state.get(key)
    if layer is not None:
        try:
            m.remove_layer(layer)
        except Exception:
            pass
        state[key] = None

def classify_chla(chla_image):
    # 严格4段分类
    classified = (
        chla_image.where(chla_image.lt(2), 1)           # Healthy
                  .where(chla_image.gte(2).And(chla_image.lt(5)), 2)   # Good
                  .where(chla_image.gte(5).And(chla_image.lt(20)), 3)  # Warning
                  .where(chla_image.gte(20), 4)          # Danger
    )
    return classified


def add_legend_to_image(img_bytes):
    img = Image.open(BytesIO(img_bytes)).convert("RGBA")
    w, h = img.size
    
    # Legend参数
    legend_w, legend_h = 200, 60
    margin = 10
    bar_h = 15
    
    # 创建legend图层
    legend = Image.new("RGBA", (legend_w, legend_h), (255, 255, 255, 200))
    draw = ImageDraw.Draw(legend)
    
    # 画色条
    colors = [
        (0x2d,0x00,0x4b), (0x4a,0x00,0x80), (0x00,0x00,0xaa),
        (0x00,0x55,0xff), (0x00,0xaa,0xff), (0x00,0xff,0xee),
        (0x00,0xff,0x88), (0xaa,0xff,0x00), (0xff,0xee,0x00),
        (0xff,0xaa,0x00), (0xff,0x44,0x00), (0xcc,0x00,0x00),
        (0x66,0x00,0x00)
    ]
    bar_w = legend_w - 20
    seg_w = bar_w / len(colors)
    for i, c in enumerate(colors):
        x0 = 10 + int(i * seg_w)
        x1 = 10 + int((i+1) * seg_w)
        draw.rectangle([x0, 10, x1, 10+bar_h], fill=c+(255,))
    
    # 画边框
    draw.rectangle([10, 10, 10+bar_w, 10+bar_h], outline=(100,100,100,255))
    
    # 标注文字
    draw.text((10, 30), "0", fill=(0,0,0,255))
    draw.text((legend_w//2 - 5, 30), "10", fill=(0,0,0,255))
    draw.text((legend_w - 25, 30), "20 µg/L", fill=(0,0,0,255))
    draw.text((10, 45), "Chl-a", fill=(0,0,0,255))
    
    # 贴到图片右下角
    pos = (w - legend_w - margin, h - legend_h - margin)
    img.paste(legend, pos, legend)
    
    output = BytesIO()
    img.save(output, format="PNG")
    return output.getvalue()


# ====================
# Run handler
# ====================
import threading

def on_run(b):
    geom = state["aoi_geom"]
    if geom is None:
        status.value = "<b style='color:red'>⚠️ Draw an AOI first.</b>"
        return
    
    status.value = "<b>Running… querying satellite images</b>"
    
    def run_ee():
        import asyncio
        asyncio.set_event_loop(asyncio.new_event_loop())
        try:
            aoi = ee.Geometry(geom)
            ic = build_s2_collection(aoi, date_picker.value, window_days.value, cloud_pct.value)
            
            n = int(ic.size().getInfo())
            if n == 0:
                status.value = "<b style='color:red'>⚠️ No scenes found.</b>"
                return

            ic_masked = ic.map(mask_clouds)
            img = ic_masked.sort("CLOUDY_PIXEL_PERCENTAGE").mosaic().clip(aoi)
            state["rgb_img"] = img
            if show_rgb.value:
                rgb_url = get_ee_tile_url(img, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 2000})
                rgb_layer = TileLayer(url=rgb_url, name="Satellite image", opacity=1.0)
                clear_layer("rgb_layer")
                m.add_layer(rgb_layer)
                state["rgb_layer"] = rgb_layer

            if show_chla.value:
                img_feat = compute_indices(img)
#                chla = lr_predict_chla(img_feat)
                chla = rf_predict_chla(img_feat)
                chla_classified = classify_chla(chla)
                state["chla_img"] = chla  
                state["chla_img_classified"] = chla_classified  
#                chla_vis = {
#                    "min": 0, "max": 10,
#                    "palette": ["2d004b","4a0080","0000aa","0055ff","00aaff",
#                                "00ffee","00ff88","aaff00","ffee00","ffaa00",
#                                "ff4400","cc0000","660000"]
#                }

#                chla_vis = {
#                   'min': 0,
#                   'max': 15,
#                   'palette': ['2166ac', '4dac26', 'f6e829', 'd7191c']
#                }

                chla_vis = {
                   'min': 1,
                   'max': 4,
                   'palette': ['2166ac', '4dac26', 'f97b06', '8B4513']
                }


                chla_url = get_ee_tile_url(chla_classified, chla_vis)
                clear_layer("chla_layer")
                chla_layer = TileLayer(url=chla_url, name="Algae aeverity (Chlorophyll-a) level", opacity=1.0)
                m.add_layer(chla_layer)
                state["chla_layer"] = chla_layer

            coords = geom["coordinates"][0]
            lats = [p[1] for p in coords]
            lons = [p[0] for p in coords]
            m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

            status.value = f"<b style='color:green'>✅ Done. Images found: {n}.</b>"

        except Exception as e:
            status.value = f"<b style='color:red'>❌ Error: {e}</b>"

    thread = threading.Thread(target=run_ee)
    thread.start()

run_btn.on_click(on_run)
def on_reset(b):
    state["aoi_geom"] = None
    state["chla_img"] = None
    state["rgb_img"] = None    
    clear_layer("rgb_layer")
    clear_layer("chla_layer")

    layers_to_remove = [l for l in m.layers if isinstance(l, Popup)]
    for l in layers_to_remove:
        m.remove_layer(l)
            
    draw.clear()
    status.value = "<b>Reset ✅ Draw a new AOI.</b>"

reset_btn.on_click(on_reset)

import requests
import base64

def on_export(b):
    if state["aoi_geom"] is None:
        status.value = "<b style='color:red'>⚠️ Run the analysis first.</b>"
        return
    
    rgb_img = state.get("rgb_img")
    chla = state.get("chla_img_classified")
    if chla is None:
        status.value = "<b style='color:red'>⚠️ No Chl-a map to export.</b>"
        return
    
    status.value = "<b>Preparing export...</b>"
    
    try:
        aoi = ee.Geometry(state["aoi_geom"])
        area_m2 = aoi.area().getInfo()
        scale = max(10, min(1000, int((area_m2 / 1e6) ** 0.5)))
        
#        chla_vis = {
#            "min": 0, "max": 10,
#            "palette": ["2d004b","4a0080","0000aa","0055ff","00aaff",
#                        "00ffee","00ff88","aaff00","ffee00","ffaa00",
#                        "ff4400","cc0000","660000"]
#        }


        chla_vis = {
                   'min': 1,
                   'max': 4,
                   'palette': ['2166ac', '4dac26', 'f97b06', '8B4513']
        }

        if rgb_img is not None and chla is not None:
            chla_vis_img = chla.visualize(**chla_vis)
            rgb_vis_img = rgb_img.visualize(**{"bands": ["B4", "B3", "B2"], "min": 0, "max": 2000})
            # 用Chl-a掩膜：水体显示Chl-a，陆地显示RGB
            water_mask = chla.mask()
            combined = rgb_vis_img.where(water_mask, chla_vis_img)
            export_img = combined
        else:
            export_img = chla.visualize(**chla_vis)

        url = export_img.getDownloadURL({
            "region": aoi,
            "scale": scale,
            "format": "PNG"
        })


        
        # 下载图片到服务器
        response = requests.get(url)
    #    img_b64 = base64.b64encode(response.content).decode("utf-8")
        img_with_legend = add_legend_to_image(response.content)
        img_b64 = base64.b64encode(img_with_legend).decode("utf-8")
        
        # 用base64 data URI触发下载
        status.value = f"""<b style='color:green'>✅ Ready!</b>
        <br><a href='data:image/png;base64,{img_b64}' 
          download='chla_map.png'>
        <button style='margin-top:6px;padding:6px 12px;background:#2b83ba;color:white;
        border:none;border-radius:4px;cursor:pointer;'>
        ⬇️ Download PNG</button></a>

        <div style='margin-top:14px;padding:10px;background:#f0f7ff;
        border-left:4px solid #2b83ba;border-radius:4px;font-family:Arial;font-size:12px;color:#444;'>
        <b>Need more data?</b><br>
        For geocoded algae severity level maps, high-resolution Chl-a GeoTIFF exports, 
        time-series Chl-a maps, or custom data requests for specific applications, 
        please contact: <a href='mailto:contact@action4water.org' style='color:#2b83ba;font-weight:bold;'>
        contact@action4water.org</a>
        </div>"""
        
    except Exception as e:
        status.value = f"<b style='color:red'>❌ Export error: {e}</b>"

export_btn.on_click(on_export)

contact_html = widgets.HTML(value="""
<div style='margin-top:12px;padding:10px;background:#f0f7ff;
border-left:4px solid #2b83ba;border-radius:4px;font-family:Arial;font-size:12px;color:#444;'>
<b>Questions or Suggestions?</b><br>
We welcome your feedback to improve this tool.<br>
<a href='mailto:contact@action4water.org' style='color:#2b83ba;font-weight:bold;'>
contact@action4water.org</a>
</div>
""")

def on_map_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    
    chla = state.get("chla_img")
    if chla is None:
        return
    
    coords = kwargs.get("coordinates")
    lat, lon = coords[0], coords[1]
    
    def query_point():
        try:
            import asyncio
            asyncio.set_event_loop(asyncio.new_event_loop())
            
            point = ee.Geometry.Point([lon, lat])
            value = chla.sample(point, 10).first().get("chla").getInfo()
            
            if value is None:
                msg = "No data (land)"
            else:
                msg = f"Chl-a: {value:.2f} µg/L"
            
            popup = Popup(
                location=(lat, lon),
                child=widgets.HTML(f"<b>{msg}</b><br><small>({lat:.4f}, {lon:.4f})</small>"),
                close_button=True,
                auto_close=True,
                close_on_escape_key=True,
            )
            m.add_layer(popup)
            
        except Exception as e:
            pass
    
    thread = threading.Thread(target=query_point)
    thread.start()

m.on_interaction(on_map_click)
# ====================
# Legend
# ====================
legend_html = widgets.HTML(value="""
<div style='padding:10px;background:white;border:2px solid #2b83ba;border-radius:8px;font-family:Arial;font-size:12px;'>
    <b>Algae Severity (Chlorophyll-a) Level</b><br><br>
    <div style='display:flex;align-items:center;margin-bottom:4px;'>
        <div style='width:20px;height:15px;background:#2166ac;margin-right:6px;border:1px solid #999;'></div>
        <span>Healthy (&lt; 2 µg/L)</span>
    </div>
    <div style='display:flex;align-items:center;margin-bottom:4px;'>
        <div style='width:20px;height:15px;background:#4dac26;margin-right:6px;border:1px solid #999;'></div>
        <span>Good (2–5 µg/L)</span>
    </div>
    <div style='display:flex;align-items:center;margin-bottom:4px;'>
        <div style='width:20px;height:15px;background:#f97b06;margin-right:6px;border:1px solid #999;'></div>
        <span>Warning (5–15 µg/L)</span>
    </div>
    <div style='display:flex;align-items:center;margin-bottom:4px;'>
        <div style='width:20px;height:15px;background:#8B4513;margin-right:6px;border:1px solid #999;'></div>
        <span>Danger (&gt; 15 µg/L)</span>
    </div>
</div>
""")
legend_control = WidgetControl(widget=legend_html, position="bottomright")
m.add_control(legend_control)

# ====================
# Layout
# ====================
title = widgets.HTML(value="<h2 style='margin:0;font-family:Arial;color:#2b83ba;'>🌊 Northern Lakes Monitor</h2>")

controls = widgets.VBox([
    widgets.HBox([about_btn, help_btn]),
    about_popup,
    help_output,
    widgets.HTML("<hr style='margin:4px 0'>"),
    date_picker,
    window_days,
    cloud_pct,
    widgets.HTML("<hr style='margin:4px 0'>"),
    show_rgb,
    show_chla,
#    widgets.HTML("<hr style='margin:4px 0'>"),
    widgets.HBox([run_btn, reset_btn]),
    export_btn,
    status,
    contact_html,
], layout=widgets.Layout(
    width="350px",
    padding="10px",
    overflow_x="hidden",
    overflow_y="auto",
    height="auto"
))

app = widgets.VBox([
    title,
    widgets.HBox([controls, m], layout=widgets.Layout(align_items="flex-start"))
], layout=widgets.Layout(width="100%"))


#display(app)

